In [1]:
from collections import defaultdict, Counter

# -----------------------------
# Sample Training Queries
# -----------------------------
queries = [
    "best places to visit in india",
    "best places to visit in chennai",
    "best places to visit near me",
    "best places to visit during summer",
    "best tourist places to visit in india",
    "best restaurants to visit in chennai",
    "new iphone launch event",
    "new samsung phone launch",
]

N = 5
DISCOUNT = 0.75

# -----------------------------
# Vocabulary
# -----------------------------
vocab = set()

# n-gram counts
ngram_counts = {i: Counter() for i in range(1, N + 1)}

# continuation counts
continuation = defaultdict(set)

# -----------------------------
# Train Model
# -----------------------------
for sentence in queries:

    words = sentence.lower().split()

    for w in words:
        vocab.add(w)

    for n in range(1, N + 1):

        for i in range(len(words) - n + 1):

            gram = tuple(words[i:i+n])
            ngram_counts[n][gram] += 1

            if n > 1:
                prefix = gram[:-1]
                continuation[prefix].add(gram[-1])

# -----------------------------
# Unknown Word Handling
# -----------------------------
def replace_oov(words):
    return [w if w in vocab else "<OOV>" for w in words]

# -----------------------------
# Recursive Kneser-Ney
# -----------------------------
def kneser_ney(context, word):

    order = len(context) + 1

    if order == 1:
        return 1 / max(len(vocab),1)

    prefix = tuple(context)
    gram = prefix + (word,)

    count_ngram = ngram_counts[order][gram]
    count_prefix = ngram_counts[order-1][prefix]

    if count_prefix == 0:
        return kneser_ney(context[1:], word)

    first = max(count_ngram - DISCOUNT, 0) / count_prefix

    unique_follow = len(continuation[prefix])

    backoff = (DISCOUNT * unique_follow) / count_prefix

    lower = kneser_ney(context[1:], word)

    return first + backoff * lower

# -----------------------------
# Prediction
# -----------------------------
def predict(text, top_k=5):

    words = replace_oov(text.lower().split())

    context = words[-4:]

    scores = {}

    for word in vocab:
        scores[word] = kneser_ney(context, word)

    suggestions = sorted(scores.items(),
                         key=lambda x: x[1],
                         reverse=True)

    return suggestions[:top_k]

# -----------------------------
# Example
# -----------------------------
query = "best places to visit"

result = predict(query)

print("Input :", query)
print("\nSuggestions")

for word, score in result:
    print(word, "->", round(score,6))

KeyboardInterrupt: 

In [2]:
print("Hello")

Hello
